In [122]:
import pandas as pd
import re

In [123]:
farm_df = pd.read_csv("../datasets/farm.csv")
cat_df = pd.read_csv("../datasets/cat_only_rt.csv")

In [124]:
cat_df = cat_df[cat_df["year"] >= 1990]
farm_df = farm_df[farm_df["year"] <= 2022]

In [125]:
farm_df

,year,state,lambs,rams,ewes,lamb_sheep_shorn,sheep_flock,sheep_purchased
0,1990,New South Wales,4073.0,206.0,8649.0,18393.0,17123,976.0
1,1990,Northern Territory,0.0,0.0,2.0,15.0,4,15.0
2,1990,Queensland,3856.0,157.0,8095.0,17240.0,17443,1023.0
3,1990,South Australia,2803.0,145.0,5276.0,11211.0,10952,941.0
4,1990,Tasmania,641.0,34.0,1141.0,2813.0,2589,86.0
...,...,...,...,...,...,...,...,...
226,2022,Queensland,1056.0,59.0,2039.0,3267.0,3706,153.0
227,2022,South Australia,3003.0,123.0,5280.0,9578.0,8842,452.0
228,2022,Tasmania,567.0,20.0,1029.0,1804.0,1795,55.0
229,2022,Victoria,1608.0,50.0,2272.0,4236.0,4239,548.0


In [126]:
cat_df

,year,stateTerritory,ibraRegion,forest2018Status,forest2013Status,capadStatus,speciesName,occurrenceCount
960,1990,Australian Capital Territory,Australian Alps,forest,forest,PA,Felis catus,1
961,1990,Australian Capital Territory,South Eastern Highlands,non-forest,non-forest,PA,Felis catus,1
962,1990,New South Wales,NSW North Coast,forest,forest,PA,Felis catus,1
963,1990,New South Wales,NSW North Coast,forest,forest,not protected,Felis catus,5
964,1990,New South Wales,South East Corner,non-forest,forest,not protected,Felis catus,1
...,...,...,...,...,...,...,...,...
4944,2022,Western Australia,Swan Coastal Plain,non-forest,non-forest,PA,Felis catus,1
4945,2022,Western Australia,Swan Coastal Plain,non-forest,non-forest,not protected,Felis catus,5
4946,2022,Western Australia,Warren,forest,forest,PA,Felis catus,1
4947,2022,Western Australia,Warren,non-forest,non-forest,PA,Felis catus,1


In [127]:
cat_df["stateTerritory"].unique()

array(['Australian Capital Territory', 'New South Wales',
       'Northern Territory', 'Queensland', 'South Australia', 'Victoria',
       'Unknown1', 'Western Australia', 'Tasmania'], dtype=object)

In [128]:
farm_df["state"].unique()

array(['New South Wales', 'Northern Territory', 'Queensland',
       'South Australia', 'Tasmania', 'Victoria', 'Western Australia'],
      dtype=object)

In [129]:
keys = ["year", "territory"]

In [130]:
cat_df = cat_df.rename(columns={"stateTerritory": "territory"})

In [131]:
cat_df = pd.concat([cat_df, cat_df.assign(territory="All Australia")], ignore_index=True)
cat_df

,year,territory,ibraRegion,forest2018Status,forest2013Status,capadStatus,speciesName,occurrenceCount
0,1990,Australian Capital Territory,Australian Alps,forest,forest,PA,Felis catus,1
1,1990,Australian Capital Territory,South Eastern Highlands,non-forest,non-forest,PA,Felis catus,1
2,1990,New South Wales,NSW North Coast,forest,forest,PA,Felis catus,1
3,1990,New South Wales,NSW North Coast,forest,forest,not protected,Felis catus,5
4,1990,New South Wales,South East Corner,non-forest,forest,not protected,Felis catus,1
...,...,...,...,...,...,...,...,...
7973,2022,All Australia,Swan Coastal Plain,non-forest,non-forest,PA,Felis catus,1
7974,2022,All Australia,Swan Coastal Plain,non-forest,non-forest,not protected,Felis catus,5
7975,2022,All Australia,Warren,forest,forest,PA,Felis catus,1
7976,2022,All Australia,Warren,non-forest,non-forest,PA,Felis catus,1


In [132]:
# sum of occurrenceCount for each (year, territory)
cats_total = (
    cat_df.groupby(keys, as_index=False)["occurrenceCount"]
           .sum()
           .rename(columns={"occurrenceCount": "cats_occurrence_total"})
)

# forest/non-forest 2013 (sum of occurrenceCount)
cats_2013 = (
    cat_df.pivot_table(index=keys,
                       columns="forest2013Status",
                       values="occurrenceCount",
                       aggfunc="sum",
                       fill_value=0)
          .rename(columns=lambda c: f"cats_2013_{c.replace('-', '_')}")
          .reset_index()
)

# forest/non-forest 2018 (sum of occurrenceCount)
cats_2018 = (
    cat_df.pivot_table(index=keys,
                       columns="forest2018Status",
                       values="occurrenceCount",
                       aggfunc="sum",
                       fill_value=0)
          .rename(columns=lambda c: f"cats_2018_{c.replace('-', '_')}")
          .reset_index()
)

In [133]:
cats_state_year = (
    cats_total.merge(cats_2013, on=keys, how="left")
              .merge(cats_2018, on=keys, how="left")
)

In [134]:
cats_state_year

,year,territory,cats_occurrence_total,cats_2013_forest,cats_2013_non_forest,cats_2018_forest,cats_2018_non_forest
0,1990,All Australia,217,104,113,115,102
1,1990,Australian Capital Territory,3,1,2,1,2
2,1990,New South Wales,69,48,21,52,17
3,1990,Northern Territory,43,11,32,14,29
4,1990,Queensland,17,9,8,7,10
...,...,...,...,...,...,...,...
236,2022,Queensland,78,19,59,16,62
237,2022,South Australia,150,20,130,14,136
238,2022,Tasmania,118,39,79,45,73
239,2022,Victoria,183,115,68,116,67


In [135]:
num_cols = [c for c in farm_df.columns if c not in ["year", "state"]]

In [136]:
# results for Australia for each year
farm_all = (
    farm_df.groupby("year", as_index=False)[num_cols]
           .sum()
           .assign(state="All Australia")
)

In [137]:
farm_all = farm_all[farm_df.columns]
farm_df = pd.concat([farm_df, farm_all], ignore_index=True)
farm_df

,year,state,lambs,rams,ewes,lamb_sheep_shorn,sheep_flock,sheep_purchased
0,1990,New South Wales,4073.0,206.0,8649.0,18393.0,17123,976.0
1,1990,Northern Territory,0.0,0.0,2.0,15.0,4,15.0
2,1990,Queensland,3856.0,157.0,8095.0,17240.0,17443,1023.0
3,1990,South Australia,2803.0,145.0,5276.0,11211.0,10952,941.0
4,1990,Tasmania,641.0,34.0,1141.0,2813.0,2589,86.0
...,...,...,...,...,...,...,...,...
259,2018,All Australia,9812.0,461.0,20227.0,35881.0,33268,2064.0
260,2019,All Australia,8477.0,526.0,19551.0,33262.0,31452,1860.0
261,2020,All Australia,8488.0,491.0,19202.0,32598.0,30921,1656.0
262,2021,All Australia,10326.0,477.0,19014.0,31989.0,31886,1913.0


In [138]:
farm_df = farm_df.rename(columns={"state": "territory"})

In [139]:
farm_cat_merged = farm_df.merge(cats_state_year, on=keys, how="outer")
farm_cat_merged

,year,territory,lambs,rams,ewes,lamb_sheep_shorn,sheep_flock,sheep_purchased,cats_occurrence_total,cats_2013_forest,cats_2013_non_forest,cats_2018_forest,cats_2018_non_forest
0,1990,All Australia,15931.0,818.0,33771.0,73652.0,70896.0,4915.0,217.0,104.0,113.0,115.0,102.0
1,1990,Australian Capital Territory,NaN,NaN,NaN,NaN,NaN,NaN,3.0,1.0,2.0,1.0,2.0
2,1990,New South Wales,4073.0,206.0,8649.0,18393.0,17123.0,976.0,69.0,48.0,21.0,52.0,17.0
3,1990,Northern Territory,0.0,0.0,2.0,15.0,4.0,15.0,43.0,11.0,32.0,14.0,29.0
4,1990,Queensland,3856.0,157.0,8095.0,17240.0,17443.0,1023.0,17.0,9.0,8.0,7.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
278,2022,Queensland,1056.0,59.0,2039.0,3267.0,3706.0,153.0,78.0,19.0,59.0,16.0,62.0
279,2022,South Australia,3003.0,123.0,5280.0,9578.0,8842.0,452.0,150.0,20.0,130.0,14.0,136.0
280,2022,Tasmania,567.0,20.0,1029.0,1804.0,1795.0,55.0,118.0,39.0,79.0,45.0,73.0
281,2022,Victoria,1608.0,50.0,2272.0,4236.0,4239.0,548.0,183.0,115.0,68.0,116.0,67.0


In [140]:
farm_cat_merged = farm_cat_merged[~farm_cat_merged["territory"].isin(["Australian Capital Territory", "Unknown1"])]
farm_cat_merged

,year,territory,lambs,rams,ewes,lamb_sheep_shorn,sheep_flock,sheep_purchased,cats_occurrence_total,cats_2013_forest,cats_2013_non_forest,cats_2018_forest,cats_2018_non_forest
0,1990,All Australia,15931.0,818.0,33771.0,73652.0,70896.0,4915.0,217.0,104.0,113.0,115.0,102.0
2,1990,New South Wales,4073.0,206.0,8649.0,18393.0,17123.0,976.0,69.0,48.0,21.0,52.0,17.0
3,1990,Northern Territory,0.0,0.0,2.0,15.0,4.0,15.0,43.0,11.0,32.0,14.0,29.0
4,1990,Queensland,3856.0,157.0,8095.0,17240.0,17443.0,1023.0,17.0,9.0,8.0,7.0,10.0
5,1990,South Australia,2803.0,145.0,5276.0,11211.0,10952.0,941.0,26.0,9.0,17.0,12.0,14.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
278,2022,Queensland,1056.0,59.0,2039.0,3267.0,3706.0,153.0,78.0,19.0,59.0,16.0,62.0
279,2022,South Australia,3003.0,123.0,5280.0,9578.0,8842.0,452.0,150.0,20.0,130.0,14.0,136.0
280,2022,Tasmania,567.0,20.0,1029.0,1804.0,1795.0,55.0,118.0,39.0,79.0,45.0,73.0
281,2022,Victoria,1608.0,50.0,2272.0,4236.0,4239.0,548.0,183.0,115.0,68.0,116.0,67.0


In [141]:
farm_cat_merged.to_csv("../datasets/farm_cat.csv", index=False)

In [142]:
eco_df = pd.read_csv('../datasets/eco_df_clean.csv')
eco_df

,Publication_year,Species,Common_name,Official_country,State|Province|Administrative_area,Location,Cost_estimate_per_year_2017_USD_exchange_rate,Applicable_year,Impacted_sector,Type_of_cost,Details
0,2002,Felis catus,Feral Cat,Australia,Unspecified,Diverse/Unspecified,8.252764e+05,2000,Authorities-Stakeholders,Control,Governments are estimated to spend at least $1...
1,2013,Felis catus,Feral Cat,Australia,Unspecified,Dirk Hartog Island\r\nArea (62 790 Ha),1.210041e+07,2013,Authorities-Stakeholders,Eradication,Cost of eradication if no internal fence is bu...
2,2013,Felis catus,Feral Cat,Australia,Unspecified,Dirk Hartog Island\r\nArea (62 790 Ha),2.946186e+04,2013,Authorities-Stakeholders,Control,Add +20 000 $ for material at each boundary of...
3,2004,Felis catus/Oryctolagus cuniculus/Vulpes vulpes,Cat/Red fox/Rabbit,Australia,Unspecified,Wardang Island/Currawinya National Park/Royal ...,9.543870e+03,2004,Authorities-Stakeholders,Control,Cost of the fence material. \r\nFox + Feral ca...
4,2004,Felis catus/Oryctolagus cuniculus/Vulpes vulpes,Cat/Red fox/Rabbit,Australia,Unspecified,Little Desert Lodge and Malleefowl Sanctuary/S...,8.971238e+03,2004,Authorities-Stakeholders,Control,Cost of the fence material. \r\nFox + Feral ca...
...,...,...,...,...,...,...,...,...,...,...,...
72,2016,Diverse/Unspecified,"rabbits, hares, cats",Australia,South Australia,Yorke Peninsula coastal,5.694082e+03,2016,Environment,Control,Case Study 3: Yorke Peninsula Coastal
73,2016,Diverse/Unspecified,"rabbits, hares, cats",Australia,South Australia,Yorke Peninsula coastal,5.694082e+03,2016,Environment,Control,Case Study 3: Yorke Peninsula Coastal
74,2016,Diverse/Unspecified,"rabbits, hares, cats",Australia,South Australia,Yorke Peninsula coastal,3.416449e+03,2016,Environment,Control,Case Study 3: Yorke Peninsula Coastal
75,2016,Diverse/Unspecified,"rabbits, hares, cats",Australia,South Australia,Yorke Peninsula coastal,2.277633e+03,2016,Environment,Control,Case Study 3: Yorke Peninsula Coastal


In [143]:
eco_df["State|Province|Administrative_area"].unique()

array(['Unspecified', 'Western Australia', 'Perth', 'Australia',
       'South Australia', 'New South Wales', 'Victoria', 'Queensland',
       'Tasmania'], dtype=object)

In [144]:
farm_cat_merged["territory"].unique()

array(['All Australia', 'New South Wales', 'Northern Territory',
       'Queensland', 'South Australia', 'Tasmania', 'Victoria',
       'Western Australia'], dtype=object)

In [145]:
# preparation for merge on the territory
eco_df["territory"] = eco_df["State|Province|Administrative_area"].astype(str).str.strip()
eco_df["territory"] = eco_df["territory"].replace({
    "Perth": "Western Australia",  # city -> state
    "Australia": "Unspecified",    # treat as not specified
})
eco_df = eco_df.drop(["State|Province|Administrative_area"], axis=1)

In [146]:
farm_cat_merged["year"].min()

np.int64(1990)

In [147]:
eco_df["Publication_year"].min()

np.int64(2002)

In [148]:
eco_df["Impacted_sector"].unique()

array(['Authorities-Stakeholders', 'Environment', 'Health', 'Agriculture'],
      dtype=object)

In [149]:
eco_df["Type_of_cost"].unique()

array(['Control', 'Eradication', 'Prevention', 'Control/Prevention',
       'Control/Education', 'Control/Medical care', 'Damage-loss'],
      dtype=object)

In [150]:
COST = "Cost_estimate_per_year_2017_USD_exchange_rate"

In [151]:
# cat only vs cat + others
eco_cat_only = (
    eco_df["Common_name"].astype(str).str.lower().str.contains(r"\bcat\b", na=False)
    & ~eco_df["Common_name"].astype(str).str.contains(r",|\+|/|;|\band\b", case=False, na=False)
)

eco_df["cost_cat_only"] = eco_df[COST].where(eco_cat_only, 0)
eco_df["cost_cat_plus_other"] = eco_df[COST].where(~eco_cat_only, 0)

In [152]:
# adding All Australia
eco_all = eco_df.copy()
eco_all["territory"] = "All Australia"
eco_df2 = pd.concat([eco_df, eco_all], ignore_index=True)

In [153]:
keys = ["Publication_year", "territory"]

In [154]:
# totals
eco_totals = (
    eco_df2.groupby(keys, as_index=False)
       .agg(
           eco_cost_total=(COST, "sum"),
           eco_cost_cat_only=("cost_cat_only", "sum"),
           eco_cost_cat_plus_other=("cost_cat_plus_other", "sum"),
       )
)

In [155]:
# function for column names
def slug(x):
    x = str(x).strip()
    x = re.sub(r"\s+", "_", x)
    x = re.sub(r"[^0-9a-zA-Z_]+", "_", x)
    return x.strip("_").lower()

In [156]:
# by sector (sum of total cost)
sector = (
    eco_df2.pivot_table(index=keys, columns="Impacted_sector", values=COST,
                    aggfunc="sum", fill_value=0)
)
sector.columns = [f"eco_sector_{slug(c)}" for c in sector.columns]
sector = sector.reset_index()

In [157]:
# by cost type (sum of total cost) ---
ctype = (
    eco_df2.pivot_table(index=keys, columns="Type_of_cost", values=COST,
                    aggfunc="sum", fill_value=0)
)
ctype.columns = [f"eco_type_{slug(c)}" for c in ctype.columns]
ctype = ctype.reset_index()

In [158]:
# final aggreagated eco table ---
eco_year_territory = (
    eco_totals.merge(sector, on=keys, how="left")
          .merge(ctype, on=keys, how="left")
)
eco_year_territory

,Publication_year,territory,eco_cost_total,eco_cost_cat_only,eco_cost_cat_plus_other,eco_sector_agriculture,eco_sector_authorities_stakeholders,eco_sector_environment,eco_sector_health,eco_type_control,eco_type_control_education,eco_type_control_medical_care,eco_type_control_prevention,eco_type_damage_loss,eco_type_eradication,eco_type_prevention
0,2002,All Australia,8.252764e+05,8.252764e+05,0.000000e+00,0.000000e+00,8.252764e+05,0.000000,0.000000e+00,8.252764e+05,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,2002,Unspecified,8.252764e+05,8.252764e+05,0.000000e+00,0.000000e+00,8.252764e+05,0.000000,0.000000e+00,8.252764e+05,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,2004,All Australia,1.180696e+06,7.157902e+05,4.649058e+05,0.000000e+00,1.180696e+06,0.000000,0.000000e+00,1.180696e+06,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
3,2004,Unspecified,1.180696e+06,7.157902e+05,4.649058e+05,0.000000e+00,1.180696e+06,0.000000,0.000000e+00,1.180696e+06,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,2006,All Australia,3.150959e+06,3.139774e+06,1.118536e+04,0.000000e+00,3.150959e+06,0.000000,0.000000e+00,1.118536e+04,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,3.139774e+06,0.000000e+00
5,2006,Unspecified,3.150959e+06,3.139774e+06,1.118536e+04,0.000000e+00,3.150959e+06,0.000000,0.000000e+00,1.118536e+04,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,3.139774e+06,0.000000e+00
6,2009,All Australia,2.411288e+06,0.000000e+00,2.411288e+06,0.000000e+00,2.411288e+06,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,2.411288e+06,0.000000e+00,0.000000e+00,0.000000e+00
7,2009,Unspecified,2.411288e+06,0.000000e+00,2.411288e+06,0.000000e+00,2.411288e+06,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,2.411288e+06,0.000000e+00,0.000000e+00,0.000000e+00
8,2010,All Australia,4.038918e+04,4.038918e+04,0.000000e+00,0.000000e+00,4.038918e+04,0.000000,0.000000e+00,1.116218e+04,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,2.922700e+04,0.000000e+00
9,2010,Unspecified,4.038918e+04,4.038918e+04,0.000000e+00,0.000000e+00,4.038918e+04,0.000000,0.000000e+00,1.116218e+04,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,2.922700e+04,0.000000e+00


In [159]:
num_cols = eco_year_territory.columns.difference(keys)
eco_year_territory[num_cols] = eco_year_territory[num_cols].fillna(0) # fill NaN in number cols

eco_year_territory = eco_year_territory.sort_values(keys).reset_index(drop=True)
eco_year_territory.head()

,Publication_year,territory,eco_cost_total,eco_cost_cat_only,eco_cost_cat_plus_other,eco_sector_agriculture,eco_sector_authorities_stakeholders,eco_sector_environment,eco_sector_health,eco_type_control,eco_type_control_education,eco_type_control_medical_care,eco_type_control_prevention,eco_type_damage_loss,eco_type_eradication,eco_type_prevention
0,2002,All Australia,8.252764e+05,8.252764e+05,0.000000,0.0,8.252764e+05,0.0,0.0,8.252764e+05,0.0,0.0,0.0,0.0,0.000000e+00,0.0
1,2002,Unspecified,8.252764e+05,8.252764e+05,0.000000,0.0,8.252764e+05,0.0,0.0,8.252764e+05,0.0,0.0,0.0,0.0,0.000000e+00,0.0
2,2004,All Australia,1.180696e+06,7.157902e+05,464905.757014,0.0,1.180696e+06,0.0,0.0,1.180696e+06,0.0,0.0,0.0,0.0,0.000000e+00,0.0
3,2004,Unspecified,1.180696e+06,7.157902e+05,464905.757014,0.0,1.180696e+06,0.0,0.0,1.180696e+06,0.0,0.0,0.0,0.0,0.000000e+00,0.0
4,2006,All Australia,3.150959e+06,3.139774e+06,11185.357970,0.0,3.150959e+06,0.0,0.0,1.118536e+04,0.0,0.0,0.0,0.0,3.139774e+06,0.0


In [160]:
eco_year_territory = eco_year_territory.rename(columns={"Publication_year": "year"})

In [161]:
keys = ["year", "territory"]
farm_cat_merged = farm_cat_merged[farm_cat_merged["year"] >= 2002]
farm_cat_eco_merged = farm_cat_merged.merge(eco_year_territory, on=keys, how="outer") # merging farm + cats + costs tables
farm_cat_eco_merged

,year,territory,lambs,rams,ewes,lamb_sheep_shorn,sheep_flock,sheep_purchased,cats_occurrence_total,cats_2013_forest,...,eco_sector_authorities_stakeholders,eco_sector_environment,eco_sector_health,eco_type_control,eco_type_control_education,eco_type_control_medical_care,eco_type_control_prevention,eco_type_damage_loss,eco_type_eradication,eco_type_prevention
0,2002,All Australia,10658.0,662.0,26531.0,51883.0,47861.0,3300.0,197.0,77.0,...,825276.359414,0.0,0.0,825276.359414,0.0,0.0,0.0,0.0,0.0,0.0
1,2002,New South Wales,2624.0,131.0,5996.0,11360.0,10544.0,770.0,104.0,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2002,Northern Territory,0.0,0.0,0.0,0.0,0.0,0.0,34.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2002,Queensland,1421.0,103.0,5020.0,11123.0,9707.0,584.0,24.0,17.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2002,South Australia,2148.0,137.0,5372.0,8857.0,8938.0,692.0,18.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174,2022,Queensland,1056.0,59.0,2039.0,3267.0,3706.0,153.0,78.0,19.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
175,2022,South Australia,3003.0,123.0,5280.0,9578.0,8842.0,452.0,150.0,20.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
176,2022,Tasmania,567.0,20.0,1029.0,1804.0,1795.0,55.0,118.0,39.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
177,2022,Victoria,1608.0,50.0,2272.0,4236.0,4239.0,548.0,183.0,115.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [162]:
farm_cat_eco_merged.to_csv("../datasets/farm_cat_eco.csv", index=False)

In [163]:
news_df = pd.read_csv('../datasets/news.csv')
news_df

,year,article_count,tone_weighted,all_articles,volume_intensity_mean,volume_intensity
0,2017,2014.0,-1.837383,227816569.0,0.000893,8.840446e-06
1,2018,1054.0,-1.624291,193012221.0,0.000544,5.460794e-06
2,2019,383.0,-2.575505,170537968.0,0.000232,2.245834e-06
3,2020,559.0,-2.436784,137760786.0,0.000379,4.057758e-06
4,2021,210.0,-1.724470,120343036.0,0.000171,1.745012e-06
5,2022,69.0,-2.148294,103082810.0,0.000065,6.693648e-07
6,2023,393.0,-2.858401,68084868.0,0.000656,5.772208e-06
7,2024,297.0,-0.819262,59778085.0,0.000503,4.968376e-06
8,2025,322.0,-1.579906,56019406.0,0.000650,5.748008e-06
9,2026,57.0,-5.721858,5092243.0,0.000946,1.119350e-05


In [164]:
# keeping only All Australia rows 2017+ in the farm + cats + costs
aa = farm_cat_eco_merged[
    (farm_cat_eco_merged["territory"] == "All Australia") &
    (farm_cat_eco_merged["year"] >= 2017)
].copy()

# merging with news by year
farm_cat_eco_news_merged = aa.merge(news_df, on="year", how="left")

farm_cat_eco_news_merged

,year,territory,lambs,rams,ewes,lamb_sheep_shorn,sheep_flock,sheep_purchased,cats_occurrence_total,cats_2013_forest,...,eco_type_control_medical_care,eco_type_control_prevention,eco_type_damage_loss,eco_type_eradication,eco_type_prevention,article_count,tone_weighted,all_articles,volume_intensity_mean,volume_intensity
0,2017,All Australia,10964.0,474.0,21306.0,36227.0,35475.0,1865.0,1371.0,900.0,...,0.000000e+00,0.0,0.000000e+00,230142.281055,0.0,2014.0,-1.837383,227816569.0,0.000893,8.840446e-06
1,2018,All Australia,9812.0,461.0,20227.0,35881.0,33268.0,2064.0,1657.0,1103.0,...,NaN,NaN,NaN,NaN,NaN,1054.0,-1.624291,193012221.0,0.000544,5.460794e-06
2,2019,All Australia,8477.0,526.0,19551.0,33262.0,31452.0,1860.0,1622.0,1049.0,...,0.000000e+00,0.0,0.000000e+00,130297.370226,0.0,383.0,-2.575505,170537968.0,0.000232,2.245834e-06
3,2020,All Australia,8488.0,491.0,19202.0,32598.0,30921.0,1656.0,1865.0,1331.0,...,4.034302e+09,0.0,7.940169e+06,852388.356504,0.0,559.0,-2.436784,137760786.0,0.000379,4.057758e-06
4,2021,All Australia,10326.0,477.0,19014.0,31989.0,31886.0,1913.0,930.0,749.0,...,NaN,NaN,NaN,NaN,NaN,210.0,-1.724470,120343036.0,0.000171,1.745012e-06
5,2022,All Australia,11568.0,470.0,20241.0,34247.0,34731.0,2273.0,1293.0,672.0,...,NaN,NaN,NaN,NaN,NaN,69.0,-2.148294,103082810.0,0.000065,6.693648e-07


In [165]:
farm_cat_eco_news_merged.to_csv("../datasets/farm_cat_eco_news.csv", index=False)